# 06. Model Explainability & SHAP Feature Attributions
**E-Commerce Customer Intelligence & Churn Prediction Platform**

### Interpretability Principles:
- We apply SHAP (SHapley Additive exPlanations) via `TreeExplainer` on the tuned XGBoost model.
- We distinguish between statistical association and causal claims: SHAP illustrates how a feature influences model log-odds, NOT that changing the feature alone guarantees churn prevention.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
from pathlib import Path

# Load champion model and test set
model = joblib.load("../models/champion_churn_model.joblib")
test_df = pd.read_csv("../data/processed/olist/holdout_test_set.csv")

feature_cols = [c for c in test_df.columns if c not in ['customer_id', 'is_churned']]
X_test = test_df[feature_cols]

# Compute SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[:500])

print("TreeExplainer initialized and computed for test sample.")

### 6.1 Global Feature Importance (SHAP Summary Plot)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.iloc[:500], plot_type="bar", show=False)
plt.title("Global Feature Importance (Mean Absolute SHAP Value)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 Local Customer Waterfall Explanation

In [ ]:
# Single Customer Explanation
cust_idx = 0
sample_exp = shap.Explanation(
    values=shap_values[cust_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[cust_idx].values,
    feature_names=feature_cols
)

plt.figure(figsize=(10, 5))
shap.plots.waterfall(sample_exp, max_display=8, show=False)
plt.title(f"Customer {test_df.iloc[cust_idx]['customer_id']} - Local SHAP Attribution", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Epistemic Framing & Business Caution:
- High recency and declining spending velocity push churn probability upward.
- These factors reflect symptoms of disengagement. Business interventions should address root dissatisfaction (e.g. shipping quality, product fit) rather than assuming a single feature is a silver bullet.